# Module 1 hook demo

Instructor only. Not linked from the student-facing site.

Used live during **`lectures/dsca-module-01.html`**, slides 2 ("Before we watch this") and 3 ("The five stages, already in front of you"). The lecture's own speaker notes for slide 2 say to switch to this notebook; this cell is the notebook side of that link.

Run this once before Module 1, with a real connection (a phone hotspot is fine), and save the notebook without clearing output. If the room's connection has a bad moment live, this morning's saved output is already on screen to narrate. If the connection holds, re-run the reference agent cell live for the real "watch it happen now" moment.

The naive bot needs no connection at all, it is a plain function. Only the reference agent cell calls a live API.

**First time running this notebook?** Do the one-time setup below before anything else.

## One-time setup

Do this once per machine, before running any cell below.

1. Create and select a Python environment for this notebook (in VS Code: the kernel picker top right; in Jupyter: `New > Python 3` or select an existing kernel).
2. Run the cell immediately below once, in that environment.
3. Copy `notebooks/.env.example` to `notebooks/.env` and fill in one of the two keys. See `notebooks/README.md` for where to get one and the free-credit note.

See `notebooks/README.md` for the full setup story and how this folder maps to each module.

In [ ]:
%pip install -r requirements.txt


## Part 1: the naive bot

No API, no internet. Deterministic every time, safe to run live regardless of connectivity.

In [ ]:
def naive_bot(turns):
    """A keyword-matching bot with no memory of its own prior turns.
    Each call only sees the current line, nothing before it."""
    booking = {}
    replies = []
    for turn in turns:
        words = turn.lower().split()
        if "book" in words or "table" in words:
            # crude slot fill from the current line only
            booking = {"party_size": None, "day": None, "time": None}
            for w in words:
                if w.isdigit():
                    booking["party_size"] = w
                if w in ("monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"):
                    booking["day"] = w
                if "pm" in w or "am" in w:
                    booking["time"] = w
            replies.append(f"Booked: {booking}")
        elif "actually" in words or "instead" in words or "make that" in turn.lower():
            # no memory of the earlier booking, so this line alone means nothing to it
            replies.append("Sorry, I didn't understand that. Could you start your booking again?")
        else:
            replies.append("Sorry, I didn't understand that.")
    return replies

conversation = [
    "Book me a table for 4 on Monday at 7pm",
    "actually, make that Tuesday",
]

for turn, reply in zip(conversation, naive_bot(conversation)):
    print(f"user: {turn}")
    print(f"bot:  {reply}\n")

The second line loses the whole booking. There is no state to correct, because there was never any state to begin with, just a reaction to whatever line came in last.

## Part 2: the reference agent

Same two lines. This one holds state across turns and updates only the field that changed. Needs `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` set in `.env`, and a live connection for this cell only.

In [ ]:
import json
import os
from dotenv import load_dotenv

load_dotenv()

# One field-schema, used to build whichever provider's structured-output
# request actually runs, so the two branches cannot quietly drift apart.
FIELDS = {
    "party_size": {"type": ["integer", "null"]},
    "day": {"type": ["string", "null"]},
    "time": {"type": ["string", "null"]},
}


def reference_agent(turns):
    """Holds one running state object across turns. Each new turn updates only
    the fields the user actually mentioned, the rest carry forward.

    Uses whichever provider has a key set in .env, OpenAI first if both are
    set, matching scripts/verify_setup.py in the team template. OpenAI uses
    a JSON-schema response format, Anthropic uses tool use with a required
    tool call, the two mechanisms named on the LLM API landscape slide."""
    if os.getenv("OPENAI_API_KEY"):
        provider = "openai"
    elif os.getenv("ANTHROPIC_API_KEY"):
        provider = "anthropic"
    else:
        raise RuntimeError(
            "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in notebooks/.env first, "
            "see notebooks/README.md."
        )

    if provider == "openai":
        from openai import OpenAI
        client = OpenAI()
        schema = {
            "name": "booking_state",
            "schema": {
                "type": "object",
                "properties": FIELDS,
                "required": list(FIELDS),
            },
        }
    else:
        import anthropic
        client = anthropic.Anthropic()
        tool = {
            "name": "update_booking_state",
            "description": "Record the current state of the table booking.",
            "input_schema": {
                "type": "object",
                "properties": FIELDS,
                "required": list(FIELDS),
            },
        }

    state = {"party_size": None, "day": None, "time": None}
    replies = []
    for turn in turns:
        prompt = (
            f"Current booking state: {json.dumps(state)}\n"
            f"User just said: \"{turn}\"\n"
            "Return the updated booking state. Keep any field the user "
            "did not just change."
        )
        if provider == "openai":
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_schema", "json_schema": schema},
            )
            state = json.loads(response.choices[0].message.content)
        else:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=200,
                tools=[tool],
                tool_choice={"type": "tool", "name": "update_booking_state"},
                messages=[{"role": "user", "content": prompt}],
            )
            tool_call = next(b for b in response.content if b.type == "tool_use")
            state = tool_call.input
        replies.append(f"Got it: {state}")
    return replies


for turn, reply in zip(conversation, reference_agent(conversation)):
    print(f"user: {turn}")
    print(f"bot:  {reply}\n")


Same correction, and the day updates while the party size and time carry forward untouched. That difference, state that persists and updates instead of a bot that reacts to one line at a time, is the whole course in miniature.

Next slide: the five-stage arc (`.lu-pipeline`), naming which module builds each stage of what was just shown.